# Block 3 — Time Series Data with ARIMA Models

**Goals for this block:**
- Understand and implement **ARIMA** for non-seasonal time series.
- Extend to **SARIMA** to model data with seasonal patterns.
- Use **`auto_arima`** for automatic hyperparameter tuning.

## 0. Setup & Environment

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

We load the train, validation and test from the previous block

In [ ]:
train_preprocessed = pd.read_parquet('train_preprocessed.parquet')
test_preprocessed = pd.read_parquet('test_preprocessed.parquet')

In [ ]:
# Naive Forecasting Model
naive_forecast = test_preprocessed['value'].iloc[-1]

In [ ]:
test_preprocessed["Naive_Forecast"] = test_preprocessed['value'].shift(1, fill_value=test_preprocessed['value'].iloc[0])

In [ ]:
test_preprocessed.round(2)

In [ ]:
test_preprocessed.plot(y=['value', 'Naive_Forecast'], figsize=(12,6), title='Naive Forecast vs Actuals')
plt.show()

## 1. ARIMA Models

ARIMA stands for **A**uto**R**egressive **I**ntegrated **M**oving **A**verage. It's a powerful class of models that captures temporal structures in time series data.

### Components of ARIMA:

-   **AR (p) - Autoregressive**: This component models the relationship between an observation and a number of lagged observations (past values). The parameter `p` is the order of the AR part. We can use the **Partial Autocorrelation Function (PACF)** plot to help determine `p`.

-   **I (d) - Integrated**: This component is used to make the time series stationary by using differencing. The parameter `d` is the number of times the data has been differenced. If `d=0`, we are essentially using an ARMA model.

-   **MA (q) - Moving Average**: This component models the relationship between an observation and the residual errors from a moving average model applied to lagged observations. The parameter `q` is the order of the MA part. We can use the **Autocorrelation Function (ACF)** plot to help determine `q`.

### The Process:

1.  **Visualize the data** to check for trends and seasonality.
2.  **Make the series stationary** using differencing (determines `d`).
3.  **Examine ACF and PACF plots** of the stationary series to determine the `p` and `q` values.
4.  **Build the ARIMA(p, d, q) model**.
5.  **Evaluate the model's performance** against the test set and compare it to our naive baseline.

Let's start by finding the right parameters for our model.

#### 1.  **Visualize the data** to check for trends and seasonality.

In [ ]:
train_preprocessed.plot(y=['value'], figsize=(7,3), title='Training Data Sample')

In [ ]:
train_preprocessed[:14].plot(y=['value'], figsize=(7,3), title='Training Data Sample')

#### 2.  **Make the series stationary** using differencing (trend order 1).


In [ ]:
train_preprocessed.diff(1).plot()

#### 3. **Examine ACF and PACF plots**

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

The ACF and PACF curves should be considered together to define the process. For an AR process, we expect the ACF curve to gradually decline and, at the same time, the PACF curve to show a sharp drop after p significant lags. To define an MA process, we expect the opposite from the ACF and PACF plots, i.e., the ACF should show a sharp decline after a certain number q of lags, while the PACF should show a geometric or gradually declining trend. If, on the other hand, both the ACF and PACF curves show a gradually decreasing pattern, then the ARMA process should be considered for modeling.

In [ ]:
train_differenciated = train_preprocessed.diff(1).dropna()

In [ ]:
# Step 1: Determine ARIMA parameters (p, d, q) by looking at ACF/PACF plots
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(train_differenciated, lags=20, ax=axes[0])
plot_pacf(train_differenciated, lags=20, ax=axes[1])
plt.show()

Interpretation:
- PACF (bottom plot) has a sharp cutoff after lag 1, suggesting p=1.
- ACF (top plot) has a sharp cutoff after lag 1, suggesting q=1.
- We use d=1 for differencing.
- A good starting model is ARIMA(1, 1, 1).

#### 4.  **Build the ARIMA(p, d, q) model**.

In [ ]:
# Step 2: Define and fit the ARIMA model
# order=(p, d, q)
arima_model = SARIMAX(train_preprocessed['value'], 
                      order=(1, 1, 1))
arima_results = arima_model.fit()

print(arima_results.summary())

#### 5.  **Evaluate on the Test set**.

In [ ]:
# Step 3: Make predictions
# We predict for the same number of steps as the length of our test set
n_steps = len(test_preprocessed)
arima_pred = arima_results.get_forecast(steps=n_steps)
arima_pred_mean = arima_pred.predicted_mean

# Add predictions to the test dataframe
test_preprocessed['ARIMA_Forecast'] = arima_pred_mean

# Step 4: Plot the results
test_preprocessed.plot(y=['value', 'Naive_Forecast', 'ARIMA_Forecast'], figsize=(12,6), title='ARIMA vs Naive Forecast')
plt.show()

## 2. SARIMA Models

While ARIMA is great for capturing trend, it falls short when the data has a clear **seasonal component**. This is where **SARIMA** (Seasonal AutoRegressive Integrated Moving Average) comes in.

SARIMA extends ARIMA by adding a new set of seasonal parameters: `(P, D, Q, s)`.

### Components of SARIMA:

The model is described as `SARIMA(p, d, q)(P, D, Q)s`:

-   **(p, d, q)**: These are the non-seasonal parameters, the same as in ARIMA.
    -   `p`: Non-seasonal AR order.
    -   `d`: Non-seasonal differencing.
    -   `q`: Non-seasonal MA order.

-   **(P, D, Q)s**: These are the new seasonal components.
    -   `P`: **Seasonal AR order**. This is the number of seasonal lagged observations (e.g., the value from the same time last week).
    -   `D`: **Seasonal differencing**. This is the number of times a seasonal difference is applied to the data (e.g., `value(t) - value(t-s)`).
    -   `Q`: **Seasonal MA order**. This is the number of seasonal lagged forecast errors.
    -   `s`: **The seasonal period**. This is the number of time steps in a single seasonal cycle (e.g., `s=7` for daily data with a weekly pattern, `s=12` for monthly data with a yearly pattern).

In [ ]:
train_differenciated = train_preprocessed.diff(7).diff(1).dropna()
train_differenciated.plot()

In [ ]:
# Step 1: Determine SARIMA parameters by looking at ACF/PACF plots
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(train_differenciated, lags=20, ax=axes[0])
plot_pacf(train_differenciated, lags=20, ax=axes[1])
plt.show()

In [ ]:
# Step 2: Define and fit the ARIMA model
# order=(p, d, q) 
# seasonal_order=(P, D, Q, s)
arima_model = SARIMAX(train_preprocessed['value'], 
                order=(1, 1, 1),
                seasonal_order=(0, 1, 1, 7))
arima_results = arima_model.fit()

print(arima_results.summary())

In [ ]:
# Step 3: Make predictions
# We predict for the same number of steps as the length of our test set
n_steps = len(test_preprocessed)
arima_pred = arima_results.get_forecast(steps=n_steps)
arima_pred_mean = arima_pred.predicted_mean

# Add predictions to the test dataframe
test_preprocessed['ARIMA_Forecast'] = arima_pred_mean

# Step 4: Plot the results
test_preprocessed.plot(y=['value', 'Naive_Forecast', 'ARIMA_Forecast'], figsize=(12,6), title='ARIMA vs Naive Forecast')
plt.show()

<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Train SARIMA</h2>

Apply what you've learned to the electricity dataset:
1. **Load the preprocessed data**: Load `train_preprocessed_ex.parquet` and `test_preprocessed_ex.parquet` that you created in the previous block.
2. **Select a Target**: Choose one energy source to forecast (e.g., 'Solar' or 'Wind').
3. **Analyze Stationarity**:
   - Plot the ACF and PACF of the differenced series to find suitable `(p,q)` and `(P,Q)` parameters. Find the right seasonal period.
4. **Build and Train**:
   - Create and fit a `SARIMA` model using the parameters you identified.
5. **Forecast and Visualize**:
   - Generate predictions for the test set.
   - Plot the actual values, your SARIMA forecast, and a simple Naive forecast on the same graph.
6. **Evaluate**:
   - Compare the results. Did the SARIMA model improve upon the baseline?

This exercise will solidify your understanding of building and evaluating seasonal time series models.
</div>

In [ ]:
# Load preprocessed data for exercise
train_ex = ...
test_ex = ...

In [ ]:
# Select a single energy source to model
energy_source = 'Solar'
train_univariate = ...
test_univariate = ...

In [ ]:
# Creating a naive forecast for comparison



In [ ]:
# Differentiating the training data


In [ ]:
# Determine SARIMA parameters by analyzing ACF/PACF plots



In [ ]:
# Define and fit the SARIMA model


In [ ]:
# Make predictions



## 3. Automatic Hyperparameter Tuning with `auto_arima`

Manually selecting the right orders for a SARIMA model by inspecting ACF/PACF plots is both an art and a science. It can be time-consuming and requires experience.

A more automated approach is to use a tool that searches through various combinations of parameters and selects the best model based on a statistical criterion, such as the **Akaike Information Criterion (AIC)**. A lower AIC value generally indicates a better-fitting model that doesn't overfit.

The `pmdarima` library provides the `auto_arima` function, which does exactly this. It performs an efficient search to find the optimal `(p,d,q)(P,D,Q)s` combination.

Let's apply `auto_arima` to our 'Solar' energy data.

In [ ]:
import pmdarima as pm

# Step 1: Use auto_arima to find the best SARIMA model
# We give it the training data and specify the seasonal period (m=7 for weekly seasonality on daily data)
auto_arima_model = pm.auto_arima(train_preprocessed['value'],
                                 start_p=1, start_q=1,
                                 max_p=3, max_q=3,
                                 max_d=2,
                                 start_P=1,start_Q=1,
                                 max_Q=3, max_P=3,
                                 max_D=2,
                                 m=7,              # Weekly seasonality
                                 seasonal=True,
                                 trace=True,
                                 error_action='ignore',  
                                 suppress_warnings=True, 
                                 stepwise=True)

In [ ]:
# Print the summary of the best model found
print(auto_arima_model.summary())

In [ ]:
# Step 2: Make predictions with the best model
auto_arima_pred = auto_arima_model.predict(n_periods=len(test_preprocessed))

In [ ]:
# Add the new forecast to our test dataframe
test_preprocessed['Auto_ARIMA_Forecast'] = auto_arima_pred

In [ ]:
test_preprocessed.plot(y=["value", 'Naive_Forecast', 'ARIMA_Forecast', 'Auto_ARIMA_Forecast'], 
                     figsize=(15, 7), 
                     title=f'All Forecasts')

<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Train SARIMA with Autotuning</h2>

Apply what you've learned to the electricity dataset:
1. **Run autoarima**: Set the right seasonal order.
2. **Check the resulting model**
3. **Evaluate**:
   - Compare the results. Did the SARIMA model improve upon the manual selected one?
4. **Select another Target**: Choose another energy source to forecast (e.g., 'Solar' or 'Wind').

This exercise will simplify the building of seasonal time series models.
</div>

In [ ]:
# Use auto_arima to find the best SARIMA model
auto_arima_model_ex = ...

In [ ]:
# Make predictions with the best model


In [ ]:
# Plot all forecasts together


## ✅ Summary 

### What You've Accomplished:

-   **Mastered ARIMA**: You learned the theory behind **ARIMA** models, including how to interpret ACF and PACF plots to manually select the `(p, d, q)` parameters for trend.
-   **Handled Seasonality with SARIMA**: You extended your knowledge to **SARIMA** to account for seasonal patterns, adding the `(P, D, Q, s)` parameters to your modeling toolkit.
-   **Automated Model Selection**: You discovered how to use `pmdarima`'s `auto_arima` function to automatically find the best model parameters, saving significant time and often improving accuracy.

You now have a solid foundation in classical statistical time series forecasting.